# Strategy analysis example

Debugging a strategy can be time-consuming. Freqtrade offers helper functions to visualize raw data.
The following assumes you work with SampleStrategy, data for 5m timeframe from Binance and have downloaded them into the data directory in the default location.
Please follow the [documentation](https://www.freqtrade.io/en/stable/data-download/) for more details.

## Setup

### Change Working directory to repository root

In [6]:
import os
from pathlib import Path
# 添加nest_asyncio支持，解决Jupyter中的事件循环冲突
try:
    import nest_asyncio
    nest_asyncio.apply()
    print("已应用nest_asyncio解决事件循环嵌套问题")
except ImportError:
    print("警告: 未安装nest_asyncio，在Jupyter中可能出现事件循环冲突。请运行 pip install nest_asyncio")


# Change directory
# Modify this cell to insure that the output shows the correct path.
# Define all paths relative to the project root shown in the cell output
project_root = "somedir/freqtrade"
i = 0
try:
    os.chdir(project_root)
    if not Path("LICENSE").is_file():
        i = 0
        while i < 4 and (not Path("LICENSE").is_file()):
            os.chdir(Path(Path.cwd(), "../"))
            i += 1
        project_root = Path.cwd()
except FileNotFoundError:
    print("Please define the project root relative to the current directory")
print(Path.cwd())

已应用nest_asyncio解决事件循环嵌套问题
Please define the project root relative to the current directory
/Users/imm-international/freqtrade


### Configure Freqtrade environment

In [7]:
from freqtrade.configuration import Configuration
from download_data import download_data

# Customize these according to your needs.

# Initialize empty configuration object
config = Configuration.from_files([Path.cwd()/"freqai_config.json"])
print(config)

2025-04-16 18:47:28,547 - freqtrade.configuration.load_config - INFO - Using config: /Users/imm-international/freqtrade/freqai_config.json ...

2025-04-16 18:47:28,552 - freqtrade.loggers - INFO - Enabling colorized output.

2025-04-16 18:47:28,554 - root - INFO - Logfile configured

2025-04-16 18:47:28,555 - freqtrade.loggers - INFO - Verbosity set to 0

2025-04-16 18:47:28,558 - freqtrade.configuration.configuration - INFO - Using user-data directory: /Users/imm-international/freqtrade/user_data ...

2025-04-16 18:47:28,559 - freqtrade.configuration.configuration - INFO - Using data directory: /Users/imm-international/freqtrade/user_data/data/okx ...

2025-04-16 18:47:28,561 - freqtrade.exchange.check_exchange - INFO - Checking exchange...

2025-04-16 18:47:28,570 - freqtrade.exchange.check_exchange - INFO - Exchange "okx" is officially supported by the Freqtrade development team.

2025-04-16 18:47:28,572 - freqtrade.configuration.configuration - INFO - Using pairlist from configuration.

{'exchange': {'name': 'okx', 'key': '', 'secret': '', 'ccxt_config': {'enableRateLimit': True, 'rateLimit': 100}, 'ccxt_async_config': {'enableRateLimit': True, 'rateLimit': 100}, 'pair_whitelist': ['BTC/USDT', 'ETH/USDT'], 'pair_blacklist': []}, 'dataformat_ohlcv': 'feather', 'dataformat_trades': 'feather', 'datadir': PosixPath('/Users/imm-international/freqtrade/user_data/data/okx'), 'dry_run': True, 'trading_mode': <TradingMode.SPOT: 'spot'>, 'stake_currency': 'USDT', 'stake_amount': 'unlimited', 'max_open_trades': 3, 'fiat_display_currency': 'USD', 'strategy': 'FreqAIStrategy', 'strategy_path': 'strategies', 'freqaimodel': 'LightGBMClassifier', 'timeframe': '15m', 'timerange': '20220101-20230331', 'timeframes': ['15m', '1h', '4h', '1d'], 'new_pairs_days': 30, 'freqai': {'enabled': True, 'purge_old_models': 2, 'train_period_days': 30, 'backtest_period_days': 7, 'identifier': 'lightgbm_classifier', 'feature_parameters': {'include_timeframes': ['15m', '1h', '4h'], 'include_corr_pairli

In [8]:
# Optionally (recommended), use existing configuration file
# config = Configuration.from_files(["user_data/config.json"])
# download_data(Path.cwd()/"freqai_config.json")

# 下载数据
success = download_data(config)
if not success:
    print("脚本无法自动下载数据，请按照上面的提示手动下载。")
else:
    print("数据下载成功，现在可以运行FreqAI训练了。")
# print('数据下载完成！')
# Define some constants
config["timeframe"] = "15m"
# Name of the strategy class
# config["strategy"] = "SampleStrategy"
# Location of the data
data_location = config["datadir"]
# Pair to analyze - Only use one pair here
pair = "BTC/USDT"

2025-04-16 18:47:28,606 - freqtrade.exchange.exchange - INFO - Instance is running with dry_run enabled

2025-04-16 18:47:28,609 - freqtrade.exchange.exchange - INFO - Using CCXT 4.4.73

2025-04-16 18:47:28,610 - freqtrade.exchange.exchange - INFO - Applying additional ccxt config: {'enableRateLimit': True, 'rateLimit': 100}

2025-04-16 18:47:28,619 - freqtrade.exchange.exchange - INFO - Applying additional ccxt config: {'enableRateLimit': True, 'rateLimit': 100}

2025-04-16 18:47:28,631 - freqtrade.exchange.exchange - INFO - Applying additional ccxt config: {'enableRateLimit': True, 'rateLimit': 100}

...

2025-04-16 18:47:30,557 - freqtrade.data.history.history_utils - INFO - Download history data for "ETH/USDT", 4h, spot and store in /Users/imm-international/freqtrade/user_data/data/okx. From 
2023-04-11T00:00:00 to 2023-03-31T00:00:00

2025-04-16 18:47:30,563 - freqtrade.data.history.history_utils - INFO - Downloaded data for ETH/USDT with length 0.

2025-04-16 18:47:30,577 - freqtrade.data.history.history_utils - INFO - Download history data for "ETH/USDT", 1d, spot and store in /Users/imm-international/freqtrade/user_data/data/okx. From 
2023-05-06T00:00:00 to 2023-03-31T00:00:00

2025-04-16 18:47:30,583 - freqtrade.data.history.history_utils - INFO - Downloaded data for ETH/USDT with length 0.

数据下载成功，现在可以运行FreqAI训练了。


In [9]:
# Load data using values set above
from freqtrade.data.history import load_pair_history
from freqtrade.enums import CandleType


candles = load_pair_history(
    datadir=data_location,
    timeframe=config["timeframe"],
    pair=pair,
    data_format="feather",  # Make sure to update this to your data
    candle_type=CandleType.SPOT,
)

# Confirm success
print(f"Loaded {len(candles)} rows of data for {pair} from {data_location}")
candles.head(100)

Loaded 43594 rows of data for BTC/USDT from /Users/imm-international/freqtrade/user_data/data/okx


,date,open,high,low,close,volume
0,2022-01-01 00:00:00+00:00,46218.3,46518.0,46216.2,46336.2,267.489110
1,2022-01-01 00:15:00+00:00,46336.3,46414.1,46240.3,46374.3,69.845254
2,2022-01-01 00:30:00+00:00,46374.4,46676.1,46360.0,46617.1,93.220566
3,2022-01-01 00:45:00+00:00,46617.1,46742.0,46578.6,46654.3,74.510431
4,2022-01-01 01:00:00+00:00,46655.1,46775.6,46578.5,46775.5,196.431324
...,...,...,...,...,...,...
95,2022-01-01 23:45:00+00:00,47506.5,47816.3,47506.0,47731.4,189.352234
96,2022-01-02 00:00:00+00:00,47731.5,47752.2,47478.0,47521.5,312.837517
97,2022-01-02 00:15:00+00:00,47521.5,47554.5,47381.7,47544.0,162.502742
98,2022-01-02 00:30:00+00:00,47543.9,47697.0,47501.8,47640.7,208.028174


## Load and run strategy
* Rerun each time the strategy file is changed

In [10]:
# Load strategy using values set above
from freqtrade.data.dataprovider import DataProvider
from freqtrade.resolvers import StrategyResolver


strategy = StrategyResolver.load_strategy(config)
strategy.dp = DataProvider(config, None, None)
strategy.ft_bot_start()

# Generate buy/sell signals using strategy
df = strategy.analyze_ticker(candles, {"pair": pair})
df.tail()

2025-04-16 18:47:30,725 - freqtrade.resolvers.iresolver - INFO - Using resolved strategy FreqAIStrategy from '/Users/imm-international/freqtrade/strategies/freqai_strategy.py'...

2025-04-16 18:47:30,727 - freqtrade.strategy.hyper - INFO - Found no parameter file.

2025-04-16 18:47:30,730 - freqtrade.resolvers.strategy_resolver - INFO - Override strategy 'timeframe' with value in config file: 15m.

2025-04-16 18:47:30,732 - freqtrade.resolvers.strategy_resolver - INFO - Override strategy 'stake_currency' with value in config file: USDT.

2025-04-16 18:47:30,733 - freqtrade.resolvers.strategy_resolver - INFO - Override strategy 'stake_amount' with value in config file: unlimited.

...

2025-04-16 18:47:30,798 - freqtrade.freqai.data_drawer - INFO - Could not find existing datadrawer, starting from scratch

2025-04-16 18:47:30,799 - freqtrade.freqai.data_drawer - INFO - Could not find existing historic_predictions, starting from scratch

2025-04-16 18:47:30,801 - freqtrade.freqai.freqai_interface - INFO - Set fresh train queue from whitelist. Queue: ['BTC/USDT', 'ETH/USDT']

2025-04-16 18:47:30,803 - freqtrade.strategy.interface - INFO - Downloading all training data for all pairs in whitelist and corr_pairlist, this may take a while if the data is not already on disk.

OperationalException: No exchange object found.

### Display the trade details

* Note that using `data.head()` would also work, however most indicators have some "startup" data at the top of the dataframe.
* Some possible problems
    * Columns with NaN values at the end of the dataframe
    * Columns used in `crossed*()` functions with completely different units
* Comparison with full backtest
    * having 200 buy signals as output for one pair from `analyze_ticker()` does not necessarily mean that 200 trades will be made during backtesting.
    * Assuming you use only one condition such as, `df['rsi'] < 30` as buy condition, this will generate multiple "buy" signals for each pair in sequence (until rsi returns > 29). The bot will only buy on the first of these signals (and also only if a trade-slot ("max_open_trades") is still available), or on one of the middle signals, as soon as a "slot" becomes available.  


In [ ]:
# Report results
print(f"Generated {df['enter_long'].sum()} entry signals")
data = df.set_index("date", drop=False)
data.tail()

## Load existing objects into a Jupyter notebook

The following cells assume that you have already generated data using the cli.  
They will allow you to drill deeper into your results, and perform analysis which otherwise would make the output very difficult to digest due to information overload.

### Load backtest results to pandas dataframe

Analyze a trades dataframe (also used below for plotting)

In [ ]:
from freqtrade.data.btanalysis import load_backtest_data, load_backtest_stats


# if backtest_dir points to a directory, it'll automatically load the last backtest file.
backtest_dir = config["user_data_dir"] / "backtest_results"
# backtest_dir can also point to a specific file
# backtest_dir = (
#   config["user_data_dir"] / "backtest_results/backtest-result-2020-07-01_20-04-22.json"
# )

In [ ]:
# You can get the full backtest statistics by using the following command.
# This contains all information used to generate the backtest result.
stats = load_backtest_stats(backtest_dir)

strategy = "SampleStrategy"
# All statistics are available per strategy, so if `--strategy-list` was used during backtest,
# this will be reflected here as well.
# Example usages:
print(stats["strategy"][strategy]["results_per_pair"])
# Get pairlist used for this backtest
print(stats["strategy"][strategy]["pairlist"])
# Get market change (average change of all pairs from start to end of the backtest period)
print(stats["strategy"][strategy]["market_change"])
# Maximum drawdown ()
print(stats["strategy"][strategy]["max_drawdown_abs"])
# Maximum drawdown start and end
print(stats["strategy"][strategy]["drawdown_start"])
print(stats["strategy"][strategy]["drawdown_end"])


# Get strategy comparison (only relevant if multiple strategies were compared)
print(stats["strategy_comparison"])

In [ ]:
# Load backtested trades as dataframe
trades = load_backtest_data(backtest_dir)

# Show value-counts per pair
trades.groupby("pair")["exit_reason"].value_counts()

## Plotting daily profit / equity line

In [ ]:
# Plotting equity line (starting with 0 on day 1 and adding daily profit for each backtested day)

import pandas as pd
import plotly.express as px

from freqtrade.configuration import Configuration
from freqtrade.data.btanalysis import load_backtest_stats


# strategy = 'SampleStrategy'
# config = Configuration.from_files(["user_data/config.json"])
# backtest_dir = config["user_data_dir"] / "backtest_results"

stats = load_backtest_stats(backtest_dir)
strategy_stats = stats["strategy"][strategy]

df = pd.DataFrame(columns=["dates", "equity"], data=strategy_stats["daily_profit"])
df["equity_daily"] = df["equity"].cumsum()

fig = px.line(df, x="dates", y="equity_daily")
fig.show()

### Load live trading results into a pandas dataframe

In case you did already some trading and want to analyze your performance

In [ ]:
from freqtrade.data.btanalysis import load_trades_from_db


# Fetch trades from database
trades = load_trades_from_db("sqlite:///tradesv3.sqlite")

# Display results
trades.groupby("pair")["exit_reason"].value_counts()

## Analyze the loaded trades for trade parallelism
This can be useful to find the best `max_open_trades` parameter, when used with backtesting in conjunction with a very high `max_open_trades` setting.

`analyze_trade_parallelism()` returns a timeseries dataframe with an "open_trades" column, specifying the number of open trades for each candle.

In [ ]:
from freqtrade.data.btanalysis import analyze_trade_parallelism


# Analyze the above
parallel_trades = analyze_trade_parallelism(trades, "5m")

parallel_trades.plot()

## Plot results

Freqtrade offers interactive plotting capabilities based on plotly.

In [ ]:
from freqtrade.plot.plotting import generate_candlestick_graph


# Limit graph period to keep plotly quick and reactive

# Filter trades to one pair
trades_red = trades.loc[trades["pair"] == pair]

data_red = data["2019-06-01":"2019-06-10"]
# Generate candlestick graph
graph = generate_candlestick_graph(
    pair=pair,
    data=data_red,
    trades=trades_red,
    indicators1=["sma20", "ema50", "ema55"],
    indicators2=["rsi", "macd", "macdsignal", "macdhist"],
)

In [ ]:
# Show graph inline
# graph.show()

# Render graph in a separate window
graph.show(renderer="browser")

## Plot average profit per trade as distribution graph

In [ ]:
import plotly.figure_factory as ff


hist_data = [trades.profit_ratio]
group_labels = ["profit_ratio"]  # name of the dataset

fig = ff.create_distplot(hist_data, group_labels, bin_size=0.01)
fig.show()

Feel free to submit an issue or Pull Request enhancing this document if you would like to share ideas on how to best analyze the data.